# ECGFounder: review training and evaluate the best checkpoint three ways

Select **Python (SYMILE-DrFuse)** and run the setup cells first. Then run each
of the three test cells, or use **Run All**. This notebook performs testing only;
it does not train or update model weights.

All conditions use the same saved multimodal checkpoint, final prediction head,
464 clinical test admissions and known labels. Excluded modalities are not loaded
or encoded. Each test has its own output cell with macro AP/AUROC and scores for
all 14 findings, including positive, negative, uncertain and unreported counts.

NaN/−1 targets are ignored. Undefined metrics stay NaN and are excluded from macro
averages. AP=1 on a positive-only finding does not establish discrimination.
Choose a completed checkpoint; do not run an evaluation while that file is being written.

The training-history section matches Pipeline2: it shows validation scores by epoch,
the saved checkpoint location and a True/False marker for the best checkpoint epoch.

In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import lightning.pytorch as pl
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import DataLoader

candidates = [candidate for parent in (Path.cwd(), *Path.cwd().parents)
              for candidate in (parent, parent / "Pipeline2_ECGFounder")]
PROJECT_DIR = next(path.resolve() for path in candidates
                   if path.name == "Pipeline2_ECGFounder"
                   and (path / "models/ecg_founder.py").is_file())
sys.path.insert(0, str(PROJECT_DIR))
for package in ("models", "utils"):
    if package in sys.modules:
        assert Path(sys.modules[package].__file__).resolve().parent == PROJECT_DIR / package, (
            "Restart the kernel: another pipeline version has already been imported."
        )

from models import MainTrainer
from utils import TARGETS, TEST_CONDITIONS, SymileMIMICDataset
from utils.evaluation import save_comparison
print("Project:", PROJECT_DIR)

## Select the training run and locate its best checkpoint

Change `RUN_DIR` to review a different run. As in Pipeline2, saved test-result
metadata identifies the selected checkpoint when available; otherwise the latest
`best*.ckpt` file is selected. The displayed path makes the choice explicit.

This is our trained multimodal checkpoint, not the official ECGFounder base
weights. The True/False checks below show whether the required paths exist.

In [ ]:
RUN_DIR = PROJECT_DIR / "runs_complete" / "version_0"
result_path = RUN_DIR / "test_results_all.pt"
if result_path.is_file():
    payload = torch.load(result_path, map_location="cpu", weights_only=False)
    CHECKPOINT_PATH = Path(payload["best_checkpoint"])
    del payload
else:
    candidates = list((RUN_DIR / "checkpoints").glob("best*.ckpt"))
    CHECKPOINT_PATH = (max(candidates, key=lambda path: path.stat().st_mtime)
                       if candidates else RUN_DIR / "checkpoints/best.ckpt")
# To select a specific checkpoint manually, uncomment and edit this line:
# CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "best-v1.ckpt"
METRICS_PATH = RUN_DIR / "metrics.csv"
metrics = pd.read_csv(METRICS_PATH) if METRICS_PATH.is_file() else pd.DataFrame()
DATASET_ROOT = PROJECT_DIR.parent / "symile-mimic/symile-mimic-aws"
BATCH_SIZE = 8

display(pd.DataFrame([
    {"item": "Training run", "path": str(RUN_DIR), "exists": RUN_DIR.is_dir()},
    {"item": "Validation metrics", "path": str(METRICS_PATH), "exists": METRICS_PATH.is_file()},
    {"item": "Best checkpoint", "path": str(CHECKPOINT_PATH), "exists": CHECKPOINT_PATH.is_file()},
]))

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Set CHECKPOINT_PATH to your saved best checkpoint: {CHECKPOINT_PATH}")

# Load only your own trusted checkpoint. Keep optimizer tensors memory-mapped on CPU.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False, mmap=True)
hparams = dict(checkpoint["hyper_parameters"])
if hparams.get("training_protocol") != "ecgfounder_complete_inputs_three_tests_v1":
    raise ValueError("Choose a checkpoint from Pipeline2_ECGFounder, not the older Pipeline2.")
label_names = list(hparams.pop("label_names"))
if label_names != TARGETS:
    raise ValueError("The checkpoint target names or order differ from our 14 targets.")

# Initialization loads the official base weights before restoring our trained weights.
# Use the local path even if the training checkpoint was moved from another machine.
hparams["ecg_checkpoint"] = PROJECT_DIR.parent / "ecgfounder/checkpoint/12_lead_ECGFounder.pth"
pl.seed_everything(hparams["seed"])
torch.set_num_threads(2)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

module = MainTrainer(args=Namespace(**hparams), label_names=label_names)
module.load_state_dict(checkpoint["state_dict"], strict=True)
module.eval()
module.requires_grad_(False)
best_epoch = int(checkpoint["epoch"])
best_step = int(checkpoint["global_step"])
print("Run:", RUN_DIR)
print("Best checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch index:", checkpoint["epoch"])
print("Completed epochs at checkpoint:", checkpoint["epoch"] + 1)
print("Optimizer steps at checkpoint:", best_step)
print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")
del checkpoint

# No logger, new training run directory, checkpoint writes, or optimizer updates.
evaluator = pl.Trainer(
    accelerator="auto", devices=1,
    logger=False, enable_checkpointing=False, enable_model_summary=False,
)
results = {}

## Validation scores after each completed epoch
Lightning writes training and validation values to separate CSV rows. Select the rows containing final-model validation scores.

In [ ]:
AP_PREFIX = "val_AP" if "val_AP_avg_over_dxs/final" in metrics else "val_PRAUC"
ap_column = f"{AP_PREFIX}_avg_over_dxs/final"
roc_column = "val_AUROC_avg_over_dxs/final"
validation = pd.DataFrame()
if ap_column in metrics.columns:
    validation = (
        metrics.dropna(subset=[ap_column])
        .sort_values(["epoch", "step"])
        .drop_duplicates(subset="epoch", keep="last")
        .copy()
    )
    validation["epoch"] = validation["epoch"].astype(int)
    
    history = validation[["epoch", ap_column, roc_column]].rename(columns={
        "epoch": "epoch_index",
        ap_column: "validation_macro_AP",
        roc_column: "validation_macro_AUROC",
    })
    history.insert(1, "completed_epochs", history["epoch_index"] + 1)
    history["saved_best_checkpoint"] = history["epoch_index"].eq(best_epoch)
    display(history.reset_index(drop=True).round(4))
else:
    print("This version contains test evaluations only; validation logs are in the training run.")

# Read the metrics belonging to the checkpoint, rather than the latest epoch.
matching = validation.loc[validation["epoch"].eq(best_epoch)] if not validation.empty else validation
best_row = matching.iloc[-1] if len(matching) else None
if best_row is not None:
    print(f"Best checkpoint: AP={best_row[ap_column]:.4f}, AUROC={best_row[roc_column]:.4f}")
else:
    print("This checkpoint was selected in an earlier training version. Review that version's validation logs for its scores.")

## Best checkpoint validation heads
New runs validate with all inputs available. The final head is the combined output; ECG, labs and CXR heads provide auxiliary diagnostics. These rows are different from the three final-head test conditions.

In [ ]:
if best_row is not None:
    head_scores = pd.DataFrame([
        {
            "prediction_head": head,
            "validation_macro_AP": best_row[f"{AP_PREFIX}_avg_over_dxs/{head}"],
            "validation_macro_AUROC": best_row[f"val_AUROC_avg_over_dxs/{head}"],
        }
        for head in ["final", "ecg", "labs", "cxr"]
    ])
    display(head_scores.round(4))


## Best checkpoint: all 14 findings
These are the final fused model’s per-finding validation scores. No Finding can have AP=1 while AUROC is undefined when its observed labels contain only positives.

In [ ]:
if best_row is not None:
    finding_scores = pd.DataFrame({
        "finding": TARGETS,
        "validation_AP": [best_row[f"{AP_PREFIX}_per_dx_final/{name}"] for name in TARGETS],
        "validation_AUROC": [best_row[f"val_AUROC_per_dx_final/{name}"] for name in TARGETS],
    })
    display(finding_scores.round(4))
    print("Findings included in macro AP:", finding_scores["validation_AP"].notna().sum())
    print("Findings included in macro AUROC:", finding_scores["validation_AUROC"].notna().sum())


## Original label counts
Counts distinguish explicit negatives from unknown labels. They are saved before training begins and cover train, validation and clinical test. Evaluation-only runs report test counts.

In [ ]:
path = RUN_DIR / "label_counts.csv"
if path.exists():
    counts = pd.read_csv(path)
    for split, table in counts.groupby("split", sort=False):
        print(split)
        display(table.reset_index(drop=True))
else:
    print("No label-count file in this older run.")

## Shared evaluation helper
This calls the existing trainer's test code, so metric calculations and label masking match `main.py`. Every evaluation displays a macro-score table and a separate 14-row finding table. It also retains admission-level probabilities in the returned predictions DataFrame.

In [ ]:
def evaluate_condition(condition):
    # Create only this input view; excluded modality files are never opened here.
    modalities = TEST_CONDITIONS[condition]
    dataset = SymileMIMICDataset(DATASET_ROOT, split="test", modalities=modalities)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    evaluator.test(model=module, dataloaders=loader, verbose=False)
    result = module.test_results

    # Ensure every independently run cell uses the same ordered test cohort.
    if results:
        reference = next(iter(results.values()))
        for key in ["hadm_ids", "y_gt", "label_mask"]:
            if not torch.equal(result[key], reference[key]):
                raise ValueError(f"Test conditions differ in {key}.")
    results[condition] = result

    summary = pd.DataFrame([{
        "condition": condition,
        "modalities": "+".join(modalities),
        "admissions": len(dataset),
        "macro_AP": result["ap"],
        "macro_AUROC": result["auroc"],
        "AP_findings": int(result["mlaps"].isfinite().sum()),
        "AUROC_findings": int(result["mlroc"].isfinite().sum()),
    }])
    findings = dataset.label_counts()
    findings.insert(1, "AP", result["mlaps"].numpy())
    findings.insert(2, "AUROC", result["mlroc"].numpy())
    predictions = pd.DataFrame(result["preds"].numpy(), columns=label_names)
    predictions.insert(0, "hadm_id", result["hadm_ids"].numpy())
    print("Checkpoint:", CHECKPOINT_PATH.name, "| Inputs:", modalities)
    print("Attention tokens:", result["attn_token_names"])
    return result, summary, findings, predictions

## Test 1 All modalities
CXR + ECG + labs. Attention uses the three distinct embeddings and one fused shared embedding.

In [ ]:
all_result, all_summary, all_findings, all_predictions = evaluate_condition("all")
display(all_summary.round(4))
display(all_findings.round(4))
# all_predictions contains each admission's 14 final probabilities.

## Test 2 CXR only
ECG and labs are excluded. Attention uses CXR distinct and CXR shared; predictions still come from the same final head, not the CXR auxiliary head.

In [ ]:
cxr_result, cxr_summary, cxr_findings, cxr_predictions = evaluate_condition("cxr_only")
display(cxr_summary.round(4))
display(cxr_findings.round(4))
# cxr_predictions contains each admission's 14 final probabilities.

## Test 3 ECG and labs
CXR is excluded. Attention uses ECG distinct, labs distinct, and their fused shared embedding.

In [ ]:
ecg_labs_result, ecg_labs_summary, ecg_labs_findings, ecg_labs_predictions = evaluate_condition("ecg_labs")
display(ecg_labs_summary.round(4))
display(ecg_labs_findings.round(4))
# ecg_labs_predictions contains each admission's 14 final probabilities.

## Compare the three tests
Run all three test cells above before this cell. Differences are relative to all modalities.

In [ ]:
missing = set(TEST_CONDITIONS) - set(results)
if missing:
    raise RuntimeError(f"Run these test cells first: {sorted(missing)}")
comparison = pd.concat([all_summary, cxr_summary, ecg_labs_summary], ignore_index=True)
comparison["delta_AP_vs_all"] = comparison["macro_AP"] - results["all"]["ap"]
comparison["delta_AUROC_vs_all"] = comparison["macro_AUROC"] - results["all"]["auroc"]
display(comparison.round(4))

## Optional: export tables
Set `SAVE_TABLES = True` to save the three comparison CSVs and admission-level predictions into a separate `notebook_evaluation` folder within this run. Training checkpoints are untouched.

In [ ]:
SAVE_TABLES = False
if SAVE_TABLES:
    if set(TEST_CONDITIONS) - set(results):
        raise RuntimeError("Run all three test cells before exporting.")
    OUTPUT_DIR = CHECKPOINT_PATH.parent.parent / "notebook_evaluation"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    save_comparison(results, all_findings.drop(columns=["AP", "AUROC"]), OUTPUT_DIR)
    for condition, frame in [("all", all_predictions), ("cxr_only", cxr_predictions),
                             ("ecg_labs", ecg_labs_predictions)]:
        frame.to_csv(OUTPUT_DIR / f"predictions_{condition}.csv", index=False)
    print("Saved:", OUTPUT_DIR)
else:
    print("Export disabled. Results remain available in the cells above.")